# Transfer Gaya Neural (*Neural Style Transfer*)

**Buku:** *Deep Learning dengan Python* — Bab 14, Deep Learning Generatif

Mengoptimalkan sebuah gambar agar kontennya mirip foto target dan gayanya (matriks Gram) mirip lukisan referensi.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Notebook ini **membutuhkan backend TensorFlow** karena memakai `tf.GradientTape` untuk gradien terhadap masukan.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `K.placeholder` + `K.gradients` + `scipy.optimize.fmin_l_bfgs_b` → `tf.Variable` + `tf.GradientTape` + optimizer SGD dengan `ExponentialDecay`.
- Kelas `Evaluator` (trik agar L-BFGS mendapat loss dan gradien sekaligus) tidak diperlukan lagi.
- `scipy.misc.imread/imsave` (dihapus) → `keras.utils.load_img/save_img`.
- Gambar target dan referensi diunduh otomatis.

In [ ]:
import os
# Backend wajib TensorFlow untuk notebook ini
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf
import numpy as np

## 1. Gambar target dan referensi gaya

In [ ]:
import matplotlib.pyplot as plt

jalur_target = keras.utils.get_file("paris.jpg", origin="https://i.imgur.com/F28w3Ac.jpg")
jalur_gaya = keras.utils.get_file("starry_night.jpg", origin="https://i.imgur.com/9ooB60I.jpg")

lebar_asli, tinggi_asli = keras.utils.load_img(jalur_target).size
TINGGI = 400
LEBAR = round(lebar_asli * TINGGI / tinggi_asli)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(keras.utils.load_img(jalur_target)); ax[0].set_title("Target (konten)")
ax[1].imshow(keras.utils.load_img(jalur_gaya)); ax[1].set_title("Referensi gaya")
for a in ax: a.axis("off")
plt.show()

## 2. Pra- dan pascapemrosesan VGG19

In [ ]:
from keras.applications import vgg19

def praproses(jalur):
    img = keras.utils.load_img(jalur, target_size=(TINGGI, LEBAR))
    arr = keras.utils.img_to_array(img)[None]
    return tf.convert_to_tensor(vgg19.preprocess_input(arr))

def pascaproses(x):
    x = x.reshape((TINGGI, LEBAR, 3)).copy()
    x[:, :, 0] += 103.939; x[:, :, 1] += 116.779; x[:, :, 2] += 123.68   # kembalikan rerata ImageNet
    x = x[:, :, ::-1]                                                    # BGR -> RGB
    return np.clip(x, 0, 255).astype("uint8")

## 3. Fungsi loss: konten, gaya, dan variasi total

In [ ]:
def gram(x):
    x = tf.transpose(x, (2, 0, 1))
    fitur = tf.reshape(x, (tf.shape(x)[0], -1))
    return tf.matmul(fitur, fitur, transpose_b=True)

def loss_gaya(gaya, kombinasi):
    S, C = gram(gaya), gram(kombinasi)
    kanal, ukuran = 3, TINGGI * LEBAR
    return tf.reduce_sum(tf.square(S - C)) / (4.0 * (kanal ** 2) * (ukuran ** 2))

def loss_konten(dasar, kombinasi):
    return tf.reduce_sum(tf.square(kombinasi - dasar))

def loss_variasi_total(x):
    a = tf.square(x[:, :TINGGI - 1, :LEBAR - 1, :] - x[:, 1:, :LEBAR - 1, :])
    b = tf.square(x[:, :TINGGI - 1, :LEBAR - 1, :] - x[:, :TINGGI - 1, 1:, :])
    return tf.reduce_sum(tf.pow(a + b, 1.25))

In [ ]:
model = vgg19.VGG19(weights="imagenet", include_top=False)
ekstraktor = keras.Model(model.inputs, {l.name: l.output for l in model.layers})

LAPISAN_GAYA = ["block1_conv1", "block2_conv1", "block3_conv1", "block4_conv1", "block5_conv1"]
LAPISAN_KONTEN = "block5_conv2"
BOBOT_VARIASI, BOBOT_GAYA, BOBOT_KONTEN = 1e-6, 1e-6, 2.5e-8

def hitung_loss(kombinasi, dasar, gaya):
    fitur = ekstraktor(tf.concat([dasar, gaya, kombinasi], axis=0))
    f = fitur[LAPISAN_KONTEN]
    loss = BOBOT_KONTEN * loss_konten(f[0], f[2])
    for nama in LAPISAN_GAYA:
        f = fitur[nama]
        loss += (BOBOT_GAYA / len(LAPISAN_GAYA)) * loss_gaya(f[1], f[2])
    return loss + BOBOT_VARIASI * loss_variasi_total(kombinasi)

@tf.function
def loss_dan_gradien(kombinasi, dasar, gaya):
    with tf.GradientTape() as tape:
        loss = hitung_loss(kombinasi, dasar, gaya)
    return loss, tape.gradient(loss, kombinasi)

## 4. Optimasi gambar kombinasi
Yang dilatih bukan bobot jaringan, melainkan **piksel** gambar `kombinasi`.

In [ ]:
optimizer = keras.optimizers.SGD(keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=100.0, decay_steps=100, decay_rate=0.96))

dasar = praproses(jalur_target)
gaya = praproses(jalur_gaya)
kombinasi = tf.Variable(praproses(jalur_target))     # mulai dari gambar target

ITERASI = 4000
SIMPAN_SETIAP = 500
for i in range(1, ITERASI + 1):
    loss, grad = loss_dan_gradien(kombinasi, dasar, gaya)
    optimizer.apply_gradients([(grad, kombinasi)])
    if i % SIMPAN_SETIAP == 0 or i == ITERASI:
        print(f"Iterasi {i}: loss = {float(loss):.2f}")
        keras.utils.save_img(f"gaya_iterasi_{i}.png", pascaproses(kombinasi.numpy()))

plt.figure(figsize=(10, 7)); plt.imshow(pascaproses(kombinasi.numpy())); plt.axis("off"); plt.show()

**Eksperimen:** ubah rasio `BOBOT_GAYA / BOBOT_KONTEN`. Rasio besar membuat gaya lebih dominan; rasio kecil mempertahankan struktur foto.